In [2]:
from reportlab.pdfgen import canvas
from pypdf import PdfReader, PdfWriter
import io

MAIN_GRID = 50
SUB_GRID = 10

reader = PdfReader("330474_J00.pdf")
page = reader.pages[0]

width = float(page.mediabox.width)
height = float(page.mediabox.height)

packet = io.BytesIO()
c = canvas.Canvas(packet, pagesize=(width, height))

# ---------- SUB GRID (10px light grey) ----------
c.setStrokeGray(0.9)
c.setLineWidth(0.2)

x = 0
while x <= width:
    c.line(x, 0, x, height)
    x += SUB_GRID

y = 0
while y <= height:
    c.line(0, y, width, y)
    y += SUB_GRID


# ---------- MAIN GRID (50px darker) ----------
c.setStrokeGray(0.6)
c.setLineWidth(0.5)

x = 0
while x <= width:
    c.line(x, 0, x, height)
    x += MAIN_GRID

y = 0
while y <= height:
    c.line(0, y, width, y)
    y += MAIN_GRID


# ---------- LABELS ----------
c.setFont("Helvetica", 6)
c.setFillGray(0.2)

x = 0
while x <= width:
    c.drawString(x + 2, 2, str(int(x)))
    x += MAIN_GRID

y = 0
while y <= height:
    c.drawString(2, y + 2, str(int(y)))
    y += MAIN_GRID


# origin marker
c.setStrokeGray(0)
c.setLineWidth(1)
c.line(0, 0, 80, 0)
c.line(0, 0, 0, 80)

c.drawString(5, 85, "(0,0)")

c.save()
packet.seek(0)

overlay = PdfReader(packet)

page.merge_page(overlay.pages[0])

writer = PdfWriter()
writer.add_page(page)

with open("debug_grid.pdf", "wb") as f:
    writer.write(f)